# Topic Discovery Dataset Preparation

## Objective

Prepare a clean dataset for BERTopic and embedding generation.

### Input
- cleaned_patent_data.csv

### Output
- patent_topic_dataset.csv

This notebook performs:
- Data validation
- Column selection
- Document quality checks
- Light text normalization
- Export of topic discovery dataset

In [1]:
import pandas as pd
import numpy as np

import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## Load Dataset

In [2]:
import os

print(os.path.abspath("../data/processed/cleaned_patent_data.csv"))

/Users/asawaribrijeshtalathi/Downloads/AI-Driven-Medical-Devices-Intelligent-System 3/src/data/processed/cleaned_patent_data.csv


In [3]:
DATA_PATH = "../data/processed/cleaned_patent_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")

Dataset Shape: (18352, 30)


In [4]:
df.head()

,source_url,patent_id,title,title_en,abstract,abstract_en,description,claims,filing_date,publication_date,country,assignee_original,assignee_en,ipc_codes,cpc_codes,claim_count,independent_claim_count,backward_citation_count,forward_citation_count,legal_status,detected_language,claims_clean,combined_text,abstract_length,word_count,claims_length_chars,claims_word_count,claims_length,filing_year,year
0,https://patents.google.com/patent/CN102708128A/en,CN102708128A,"Methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment","Methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment","The invention is entitled ""Methods and systems for receiving, mapping and structuring data from disparate systems in...","The invention is entitled ""Methods and systems for receiving, mapping and structuring data from disparate systems in...","The invention is entitled ""Methods and systems for receiving, mapping and structuring data from disparate systems in...",1.一种接收与卫生保健消息结构的不同版本关联的数据输入的计算机实现的方法，包括： 1. A computer-implemented method of receiving data input associated with di...,2012-02-21,2012-10-03,CN,General Electric Co,General Electric Co,['G16H10/60'],['G16H10/60'],22,4,60,0,Status,zh,A computer-implemented method of receiving data input associated with different versions of a healthcare message str...,"Methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment T...",1000,1302,9039,1132,9039,2012.0,2012
1,https://patents.google.com/patent/KR102798691B1/en,KR102798691B1,Electrocardiogram interpretating system with unified deep learning based model and rule based model,Electrocardiogram interpretating system with unified deep learning based model and rule based model,The present invention comprises an electrocardiogram measuring unit (110) that measures one or more leads of an elec...,The present invention comprises an electrocardiogram measuring unit (110) that measures one or more leads of an elec...,The present invention comprises an electrocardiogram measuring unit (110) that measures one or more leads of an elec...,"1유도 이상의 심전도를 측정하여 심전도 데이터를 생성하는 심전도 측정부; 상기 1유도 이상의 심전도 및 이에 상응하는 질환의 학습데이터셋으로 학습하여 구축된 딥러닝 알고리즘을 통해, 상기 심전도 측정부로부터 ...",2021-08-17,2023-02-24,KR,주식회사 메디컬에이아이,주식회사 메디컬에이아이,['A61B5/0006'],['A61B5/0006'],5,5,62,0,Status,ko,An electrocardiogram measuring unit that generates electrocardiogram data by measuring one or more leads of an elect...,Electrocardiogram interpretating system with unified deep learning based model and rule based model The present inve...,1352,943,4798,755,4798,2021.0,2021
2,https://patents.google.com/patent/US11913937B2/en,US11913937B2,Method and device for automatically tracking urine,Method and device for automatically tracking urine,A method of automatically tracking urine in a toilet includes creating a background image of an interior of a toilet...,A method of automatically tracking urine in a toilet includes creating a background image of an interior of a toilet...,CROSS-REFERENCE TO RELATED PATENT APPLICATIONS This application is a Continuation of U.S. patent application Ser. No...,"1. A method of automatically tracking urine in a toilet, the method comprising: creating a background image of an in...",2021-08-13,2021-12-02,US,Shanghai Kohler Electronics Ltd,Shanghai Kohler Electronics Ltd,['G01N33/493'],['G01N33/493'],20,3,81,0,Status,en,"A method of automatically tracking urine in a toilet, the method comprising: creating a background image of an inter...",Method and device for automatically tracking urine A method of automatically tracking urine in a toilet includes cre...,759,1229,6604,1097,6604,2021.0,2021
3,https://patents.google.com/patent/JP7095207B2/en,JP7095207B2,X-ray detector with independent sleepable processor,X-ray detector with independent sleepable processor,NaN,NaN,NaN,入射するＸ線に応答して複数の信号を生成するよう構成されるデジタルイメージセンサと、 前記デジタルイメ

In [5]:
print("Total rows:", len(df))

Total rows: 18352


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18352 entries, 0 to 18351
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   source_url               18352 non-null  object 
 1   patent_id                18352 non-null  object 
 2   title                    18352 non-null  object 
 3   title_en                 18352 non-null  object 
 4   abstract                 17824 non-null  object 
 5   abstract_en              17824 non-null  object 
 6   description              17877 non-null  object 
 7   claims                   18274 non-null  object 
 8   filing_date              18352 non-null  object 
 9   publication_date         18352 non-null  object 
 10  country                  18352 non-null  object 
 11  assignee_original        18336 non-null  object 
 12  assignee_en              18336 non-null  object 
 13  ipc_codes                18352 non-null  object 
 14  cpc_codes             

In [7]:
df.columns.tolist()

['source_url',
 'patent_id',
 'title',
 'title_en',
 'abstract',
 'abstract_en',
 'description',
 'claims',
 'filing_date',
 'publication_date',
 'country',
 'assignee_original',
 'assignee_en',
 'ipc_codes',
 'cpc_codes',
 'claim_count',
 'independent_claim_count',
 'backward_citation_count',
 'forward_citation_count',
 'legal_status',
 'detected_language',
 'claims_clean',
 'combined_text',
 'abstract_length',
 'word_count',
 'claims_length_chars',
 'claims_word_count',
 'claims_length',
 'filing_year',
 'year']

In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
source_url,18352,18352,https://patents.google.com/patent/CN102708128A/en,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patent_id,18352,18352,CN102708128A,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,18352,16563,Oral care implement,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title_en,18352,16563,Oral care implement,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
abstract,17824,16350,FIELD: medicine.,189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
abstract_en,17824,16350,FIELD: medicine.,189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,17877,17429,A method of establishing a referral network for the enrollment of individuals in a geriatric healthcare insurance pl...,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claims,18274,18002,"1. A system for automatically detecting medical devices positioned within a room of a healthcare facility, the syste...",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
filing_date,18352,4644,2013-11-26,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
publication_date,18352,3165,2013-12-25,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
DATA_PATH = "../data/processed/cleaned_patent_data.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

(18352, 30)


In [10]:
# Create topic dataset
topic_df = df[
    [
        "patent_id",
        "title",
        "combined_text",
        "filing_year",
        "country",
        "assignee_en",
        "cpc_codes",
        "ipc_codes"
    ]
].copy()

In [11]:
topic_df.shape

(18352, 8)

In [12]:
# Check missing values in selected columns
topic_df.isnull().sum()

patent_id         0
title             0
combined_text     0
filing_year       0
country           0
assignee_en      16
cpc_codes         0
ipc_codes         0
dtype: int64

In [13]:
# Check duplicate text

# Even though patent IDs are unique, two different patents may have identical text.

# Run:

topic_df["combined_text"].duplicated().sum()

np.int64(234)

In [14]:
topic_df = topic_df.drop_duplicates(
    subset="combined_text"
)

In [15]:
# Create document length feature
topic_df["document_length"] = (
    topic_df["combined_text"]
    .str.len()
)

In [16]:
topic_df["document_length"].describe()

count     18118.000000
mean       8622.881720
std        9256.684837
min         303.000000
25%        4545.000000
50%        7140.000000
75%       10327.750000
max      548084.000000
Name: document_length, dtype: float64

In [17]:
# Inspect short documents
topic_df.sort_values(
    "document_length"
).head(10)

,patent_id,title,combined_text,filing_year,country,assignee_en,cpc_codes,ipc_codes,document_length
8796,KR20210141823A,Method and apparatus for the treatment of pain in patients with intervertebral back pain,Method and apparatus for the treatment of pain in patients with intervertebral back pain Disclosed is an exercise pr...,2020.0,KR,밸류앤드트러스트(주),['G16H20/30'],['G16H20/30'],303
11249,WO2010132393A3,System and method for matching health care providers with consumers,System and method for matching health care providers with consumers A system and method for providing both healthcar...,2010.0,WO,Individual,['G06Q30/02'],['G06Q30/02'],318
13009,BR112017015045A2,health care data exchange method and system,health care data exchange method and system healthcare data system that is part of a scalable technology core that c...,2016.0,BR,Pricewaterhousecoopers Llp,['G16H10/60'],['G16H10/60'],320
16235,MX386749B,SYSTEM AND PROCEDURE FOR DATA EXCHANGE IN HEALTH CARE.,SYSTEM AND PROCEDURE FOR DATA EXCHANGE IN HEALTH CARE. A health data system that is part of a scalable technology co...,2016.0,MX,PwC Product Sales LLC,['G16H10/60'],['G16H10/60'],344
14862,EP2458518A3,Remote healthcare system and healthcare method using the same,Remote healthcare system and healthcare method using the same A remote healthcare system and healthcare method using...,2011.0,EP,Samsung Electronics Co Ltd,['G16H10/60'],['G16H10/60'],399
15406,WO2010118327A3,Targeted health care content delivery system,"Targeted health care content delivery system In one example, a method of providing targeted health care content to a...",2010.0,WO,Fusion Global LLC,['G06Q10/10'],['G06Q10/10'],416
16841,WO2010028237A3,Health care data management,Health care data management Health care data is stored in memory accessible to a server. The server allows users to ...,2009.0,WO,MLP TECHNOLOGY Inc F/K/A MY LIFEPLAN Inc,['G06F16/9537'],['G06F16/9537'],432
12069,WO2013188569A3,System and method for reducing healthcare-associated infections,System and method for reducing healthcare-associated infections Systems and methods for reducing the incidence of he...,2013.0,WO,Kimberly Clark Corp,['G06Q10/0635'],['G06Q10/0635'],458
15557,WO2013055751A3,Systems and methods for health care credit transactions,"Systems and methods for health care credit transactions Systems, methods, and computer program media for creating, m...",2012.0,WO,STAGE 5 INNOVATION LLC,['G06Q40/08'],['G06Q40/08'],464
16916,MX2013000201A,METHODS TO SUPPLY A HEALTH CARE ASSETS BY MEANS OF PERSONAL CARE ITEMS THAT INCLUDE A FILAMENT.,METHODS TO SUPPLY A HEALTH CARE ASSETS BY MEANS OF PERSONAL CARE ITEMS THAT INCLUDE A FILAMENT. A method of providin...,2011.0,MX,Procter & Gamble,['A61K9/70'],['A61K9/70'],476


In [18]:
# Remove very short documents

# Example:

topic_df = topic_df[
    topic_df["document_length"] >= 200
]

print(topic_df.shape)

(18118, 9)


In [19]:
# Create topic text column

# Keep your original combined_text unchanged.

# Create a separate column for BERTopic.

import re

def clean_topic_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


topic_df["topic_text"] = (
    topic_df["combined_text"]
    .apply(clean_topic_text)
)

In [20]:
# Final validation

# Check:

topic_df.head()

,patent_id,title,combined_text,filing_year,country,assignee_en,cpc_codes,ipc_codes,document_length,topic_text
0,CN102708128A,"Methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment","Methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment T...",2012.0,CN,General Electric Co,['G16H10/60'],['G16H10/60'],10155,"methods and systems for receiving, mapping and structuring data from disparate systems in a healthcare environment t..."
1,KR102798691B1,Electrocardiogram interpretating system with unified deep learning based model and rule based model,Electrocardiogram interpretating system with unified deep learning based model and rule based model The present inve...,2021.0,KR,주식회사 메디컬에이아이,['A61B5/0006'],['A61B5/0006'],6251,electrocardiogram interpretating system with unified deep learning based model and rule based model the present inve...
2,US11913937B2,Method and device for automatically tracking urine,Method and device for automatically tracking urine A method of automatically tracking urine in a toilet includes cre...,2021.0,US,Shanghai Kohler Electronics Ltd,['G01N33/493'],['G01N33/493'],7415,method and device for automatically tracking urine a method of automatically tracking urine in a toilet includes cre...
3,JP7095207B2,X-ray detector with independent sleepable processor,X-ray detector with independent sleepable processor 入射するＸ線に応答して複数の信号を生成するよう構成されるデジタルイメージセンサと、 前記デジタルイメージセンサに通信可能に結合...,2019.0,JP,ヴァレックス イメージング コーポレイション,"['G01N21/00', 'G01N23/04', 'G01N22/00', 'G01N17/00', 'G01N3/00']","['G01N21/00', 'G01N23/04', 'G01N22/00', 'G01N17/00', 'G01N3/00']",10670,x-ray detector with independent sleepable processor 入射するｘ線に応答して複数の信号を生成するよう構成されるデジタルイメージセンサと、 前記デジタルイメージセンサに通信可能に結合さ...
4,CN108538402A,"A kind of MDT consultation of doctors method, system","A kind of MDT consultation of doctors method, system The present invention discloses a kind of MDT consultation of d...",2018.0,CN,Shenzhen Zero One Cloud Medical Science And Technology Co Ltd,['G16H80/00'],['G16H80/00'],9102,"a kind of mdt consultation of doctors method, system the present invention discloses a kind of mdt consultation of d..."


In [21]:
topic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18118 entries, 0 to 18351
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   patent_id        18118 non-null  object 
 1   title            18118 non-null  object 
 2   combined_text    18118 non-null  object 
 3   filing_year      18118 non-null  float64
 4   country          18118 non-null  object 
 5   assignee_en      18102 non-null  object 
 6   cpc_codes        18118 non-null  object 
 7   ipc_codes        18118 non-null  object 
 8   document_length  18118 non-null  int64  
 9   topic_text       18118 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 1.5+ MB


In [22]:
# Save patent_topic_dataset.csv
OUTPUT_PATH = "../data/processed/patent_topic_dataset.csv"

topic_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully:")
print(OUTPUT_PATH)

Saved successfully:
../data/processed/patent_topic_dataset.csv


In [23]:
# Verify the saved file
check = pd.read_csv(
    "../data/processed/patent_topic_dataset.csv"
)

print(check.shape)

(18118, 10)
